# 06d · Modelado: Propuestas C (ST-GRU) y D (TCN Multi-Output)

| | |
|---|---|
| **Fase CRISP-DM** | 4 — Modelado |
| **Objetivo** | Implementar las Propuestas C (2.ª del Benchmarking, 4.00) y D (3.ª, 3.90) tal como las describe la literatura, para la comparación. |
| **Entradas** | `data/gold/series_filtradas.parquet` (voltaje real, calidad, kA; solo train) |
| **Salidas** | `models/modelo_C.pt`, `models/modelo_D.pt`, `reports/modelado/cd_*` |

**Entrada común (según `detalle-propuestas-modelos-ml.md` y `prompt_v1.md`):** la **serie real normalizada sin filtro externo** (Min-Max por celda y para kA, ajustado con train). Los días sin dato se rellenan con el último valor anterior (solo pasado, sin fuga de información).

**Propuesta C — ST-GRU**
- **Grafo de celdas** por electrolizador: cada celda se conecta con sus **vecinas en serie** (i ± 1) y con las **4 celdas cuyo voltaje diario se movió de forma más parecida en train** (correlación de las variaciones diarias). Matriz normalizada `D^-1/2 (A + I) D^-1/2`.
- **Componente espacial:** en cada día, dos capas de convolución de grafo (GCN) mezclan cada celda con sus vecinas.
- **Componente temporal:** una GRU recorre los 30 días de cada celda (pesos compartidos entre celdas).
- **Salida:** proyección de los 180 días de todo el grafo `[B, 180, N]`.

**Propuesta D — TCN Multi-Output**
- Las celdas + kA como canales `[B, 30, N + 1]`, un modelo por electrolizador.
- Convolución de suavizado inicial + bloques residuales de **convoluciones causales dilatadas** (dilatación 1, 2, 4, 8, 16: el campo receptivo cubre los 30 días).
- **Capa densa multi-salida** que proyecta `[B, 180, N]` en paralelo.

**Tal cual la literatura:** sin máscara de saltos. Misma semilla y mismo protocolo de early stopping + reentrenamiento que los demás modelos.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

from voltcast import cleaning as cl
from voltcast import config, io
from voltcast import dataset as ds
from voltcast import mimo
from voltcast import model as mdl

config.ensure_dirs()
FIG_DIR = config.REPORTS_DIR / "modelado"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE}\nST-GRU: {config.STGRU}\nTCN: {config.TCN}")


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight")
    print(f"Figura guardada: {path.relative_to(config.BASE_DIR)}")

## 1. Serie real normalizada (Min-Max, solo train)

In [ ]:
gold = io.read_layer("gold", "series_filtradas.parquet")
train = gold[gold["split"] == "train"].copy()
train["celda"] = train["celda"].astype(str)
CELLS = sorted(train["celda"].unique())
GRUPOS = {el: [c for c in CELLS if c[0] == el] for el in ("A", "B")}
FECHAS = pd.DatetimeIndex(sorted(train["fecha"].unique()))
T_ = len(FECHAS)
CUT = T_ - config.ES_BLOCK_DAYS

VR = ds.pivot(train, "voltaje", CELLS).reindex(FECHAS)
VALIDO = ds.pivot(train, "calidad", CELLS).reindex(FECHAS).isin(cl.CALIDAD_VALIDA)
KA = {el: train[train["celda"] == GRUPOS[el][0]].set_index("fecha")["kA_z"].reindex(FECHAS).to_numpy()[:, None]
      for el in GRUPOS}

PIEZAS = {}
for el, celdas in GRUPOS.items():
    vr = VR[celdas].where(VALIDO[celdas]).to_numpy()
    csc = mimo.MinMax().fit(vr)
    ksc = mimo.MinMax().fit(KA[el])
    entrada = mimo.causal_ffill(vr)                       # último valor válido anterior (causal)
    X_tcn = mimo.scaled_inputs(entrada, KA[el], csc, ksc)  # [T, N + 1]
    Y = np.where(VALIDO[celdas].to_numpy(), csc.transform(VR[celdas].to_numpy()), np.nan)
    PIEZAS[el] = {"X_tcn": X_tcn, "Y": Y, "V": VALIDO[celdas].to_numpy(), "celdas": celdas, "csc": csc, "ksc": ksc,
                  "vr": vr}
    print(f"EL {el}: {len(celdas)} celdas | entrada TCN [T, {X_tcn.shape[1]}]")

## 2. Propuesta C: grafo de celdas

In [ ]:
def adyacencia(vr, vecinas=1, k=4):
    """Vecinas en serie (i ± vecinas) + k celdas de mayor correlación de las variaciones diarias en train."""
    N = vr.shape[1]
    A = np.zeros((N, N))
    for d in range(1, vecinas + 1):
        i = np.arange(N - d)
        A[i, i + d] = A[i + d, i] = 1
    dif = pd.DataFrame(np.diff(vr, axis=0))
    corr = dif.corr(min_periods=100).to_numpy().copy()
    np.fill_diagonal(corr, -np.inf)
    corr = np.nan_to_num(corr, nan=-np.inf)
    for i in range(N):
        for j in np.argsort(corr[i])[-k:]:
            if np.isfinite(corr[i, j]):
                A[i, j] = A[j, i] = 1
    A_hat = A + np.eye(N)
    d = 1 / np.sqrt(A_hat.sum(1))
    return A, d[:, None] * A_hat * d[None, :]


ADJ, CRUDA = [], {}
for el in GRUPOS:
    A, A_norm = adyacencia(PIEZAS[el]["vr"], config.STGRU["vecinas"], config.STGRU["k_correlacion"])
    ADJ.append(A_norm)
    CRUDA[el] = A
    print(f"EL {el}: {int(A.sum() / 2)} aristas | grado medio {A.sum(1).mean():.1f}")
ADJ = np.stack(ADJ)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (el, A) in zip(axes, CRUDA.items()):
    ax.imshow(A, cmap="Greys", interpolation="nearest")
    ax.set(title=f"EL {el}: grafo de celdas (vecinas + correlación)", xlabel="celda", ylabel="celda")
fig.tight_layout(); savefig(fig, "cd_grafo_stgru")

## 3. Entrenamiento

In [ ]:
def entrenar(data, cfg, epochs=None, val=True, log=None):
    torch.manual_seed(config.SEED)
    m = mdl.build_model(0, 0, cfg).to(DEVICE)
    vp = data.origins(CUT, T_ - 2) if val else None
    rng = (config.LOOKBACK_DAYS - 1, CUT - 1) if val else (config.LOOKBACK_DAYS - 1, T_ - 2)
    lim = CUT - 1 if val else T_ - 1
    r = mdl.fit(m, data, rng, lim, cfg, val_pairs=vp, val_target_limit=T_ - 1, epochs=epochs,
                mask_jumps=False, log=log)
    return m, r


HIST, RES = [], {}

# --- Propuesta C: un modelo para los dos grafos (índice de grupo = electrolizador) ---
X_c = {el: np.stack([np.nan_to_num(PIEZAS[el]["csc"].transform(mimo.causal_ffill(PIEZAS[el]["vr"]))),
                     np.repeat(np.nan_to_num(PIEZAS[el]["ksc"].transform(KA[el])), len(GRUPOS[el]), axis=1)],
                    axis=-1).astype(np.float32) for el in GRUPOS}  # [T, N, 2]
data_c = ds.MatrixTensors(X_c, {el: PIEZAS[el]["Y"] for el in GRUPOS}, {el: PIEZAS[el]["V"] for el in GRUPOS},
                          FECHAS, DEVICE)
cfg_c = {**config.STGRU, "adjacency": ADJ, "n_features": 2}
t = time.time()
m_es, r_es = entrenar(data_c, cfg_c, log=print)
m_c, r_fin = entrenar(data_c, cfg_c, epochs=r_es["mejor_epoca"], val=False)
RES["C"] = {"modelo": m_c, "cfg": cfg_c, "r_es": r_es, "min": (time.time() - t) / 60}
HIST += [{"modelo": "C", "grupo": "AB", "fase": f, **h} for f, r in [("early_stopping", r_es), ("reentrenamiento", r_fin)]
         for h in r["historia"]]
print(f"→ C: mejor época {r_es['mejor_epoca']}, {mdl.count_params(m_c):,} parámetros, {RES['C']['min']:.1f} min")

# --- Propuesta D: un modelo por electrolizador ---
t = time.time()
modelos_d, cfg_d, mejores = {}, {}, {}
for el in GRUPOS:
    data_d = ds.MatrixTensors({el: PIEZAS[el]["X_tcn"]}, {el: PIEZAS[el]["Y"]}, {el: PIEZAS[el]["V"]}, FECHAS, DEVICE)
    cfg = {**config.TCN, "n_in": PIEZAS[el]["X_tcn"].shape[1], "n_out": len(GRUPOS[el])}
    m_es, r_es = entrenar(data_d, cfg, log=None)
    m_d, r_fin = entrenar(data_d, cfg, epochs=r_es["mejor_epoca"], val=False)
    modelos_d[el], cfg_d[el], mejores[el] = m_d, cfg, r_es
    HIST += [{"modelo": "D", "grupo": el, "fase": f, **h} for f, r in [("early_stopping", r_es), ("reentrenamiento", r_fin)]
             for h in r["historia"]]
    print(f"D · EL {el}: mejor época {r_es['mejor_epoca']}, MSE bloque {r_es['mejor_val_mse']:.5f}, "
          f"{mdl.count_params(m_d):,} parámetros")
RES["D"] = {"modelo": mdl.GroupModels(modelos_d), "cfg": cfg_d, "r_es": mejores, "min": (time.time() - t) / 60}
print(f"→ D: {RES['D']['min']:.1f} min")
pd.DataFrame(HIST).to_csv(FIG_DIR / "cd_historial_entrenamiento.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
h = pd.DataFrame(HIST)
for (mod, grupo), g in h[h["fase"] == "early_stopping"].groupby(["modelo", "grupo"]):
    l, = ax.plot(g["epoca"], g["train_mse"], label=f"{mod}·{grupo} train")
    ax.plot(g["epoca"], g["val_mse"], ls="--", color=l.get_color(), label=f"{mod}·{grupo} bloque interno")
ax.set(yscale="log", xlabel="época", ylabel="MSE (normalizado)", title="Propuestas C y D: curvas de entrenamiento")
ax.legend(fontsize=7, ncol=2); fig.tight_layout(); savefig(fig, "cd_curvas")

**Lectura.**
- **C · ST-GRU:** entrena de forma estable y lenta (mejor época 49, ~7 min). Es un modelo muy pequeño (~32 000 parámetros) porque la GCN y la GRU comparten pesos entre todas las celdas; el grafo aporta la información de las celdas vecinas y de las de comportamiento parecido.
- **D · TCN:** converge rápido (mejores épocas 21 y 19, < 1 min) y tiene muchos más parámetros (~1,9 millones por electrolizador), casi todos en la capa densa que proyecta las 150 celdas × 180 días.
- Los MSE de esta etapa están en la escala Min-Max de cada modelo y no son comparables entre propuestas; la comparación en voltios y sobre los mismos objetivos se hace en el notebook 07.

## 4. Guardado

In [ ]:
esc = {el: {"celdas": PIEZAS[el]["csc"].to_dict(), "kA": PIEZAS[el]["ksc"].to_dict()} for el in GRUPOS}
comun = {"cells": CELLS, "features": [], "grupos": list(GRUPOS), "celdas_por_grupo": GRUPOS, "escaladores": esc,
         "senal_entrada": "voltaje real normalizado (sin filtro), relleno causal", "lookback": config.LOOKBACK_DAYS,
         "horizon": config.HORIZON_DAYS, "seed": config.SEED, "mask_post_jump": False,
         "entrenado_hasta": str(FECHAS[-1].date()), "fecha_entrenamiento": pd.Timestamp.now().isoformat(timespec="seconds")}
mdl.save_checkpoint(config.MODELS_DIR / "modelo_C.pt", RES["C"]["modelo"],
                    {**comun, "nombre": "Propuesta C (ST-GRU)", "arch": "stgru", "train_cfg": RES["C"]["cfg"],
                     "mejor_epoca": RES["C"]["r_es"]["mejor_epoca"], "tiempo_min": {"entrenamiento_min": RES["C"]["min"]}})
mdl.save_checkpoint(config.MODELS_DIR / "modelo_D.pt", RES["D"]["modelo"],
                    {**comun, "nombre": "Propuesta D (TCN Multi-Output)", "arch": "tcn", "cfg_por_grupo": RES["D"]["cfg"],
                     "train_cfg": {**RES["D"]["cfg"]["A"], "groups": list(GRUPOS)},
                     "mejor_epoca": {el: r["mejor_epoca"] for el, r in RES["D"]["r_es"].items()},
                     "tiempo_min": {"entrenamiento_min": RES["D"]["min"]}})
for n in ("C", "D"):
    p = config.MODELS_DIR / f"modelo_{n}.pt"
    print(f"✔ {p.relative_to(config.BASE_DIR)} ({p.stat().st_size / 1e6:.1f} MB)")

## 5. Verificaciones

In [ ]:
checks = {}
m, ck = mdl.load_checkpoint(config.MODELS_DIR / "modelo_C.pt", DEVICE)
x = data_c.X[:, -config.LOOKBACK_DAYS:]
g = torch.arange(2, device=DEVICE)
with torch.no_grad():
    checks["C: el checkpoint reproduce las predicciones"] = torch.allclose(m(x, g), RES["C"]["modelo"].eval()(x, g), atol=1e-5)
ck = torch.load(config.MODELS_DIR / "modelo_D.pt", map_location=DEVICE, weights_only=False)
red = mdl.GroupModels({el: mdl.build_model(0, 0, ck["cfg_por_grupo"][el]) for el in ck["grupos"]}).to(DEVICE)
red.load_state_dict(ck["state_dict"]); red.eval()
for i, el in enumerate(ck["grupos"]):
    xd = torch.as_tensor(PIEZAS[el]["X_tcn"][None, -config.LOOKBACK_DAYS:], device=DEVICE)
    with torch.no_grad():
        p1, p2 = red(xd, torch.full((1,), i, device=DEVICE)), RES["D"]["modelo"].eval()(xd, torch.full((1,), i, device=DEVICE))
    checks[f"D·{el}: el checkpoint reproduce las predicciones"] = torch.allclose(p1, p2, atol=1e-5)
    checks[f"D·{el}: salida [1, {config.HORIZON_DAYS}, {len(GRUPOS[el])}]"] = tuple(p1.shape) == (1, config.HORIZON_DAYS, len(GRUPOS[el]))
checks["solo fechas de train"] = FECHAS[-1] < gold.loc[gold["split"] == "val", "fecha"].min()
for k, v in checks.items():
    print(("✔" if v else "✘"), k)
assert all(checks.values())

## Conclusión

- Se implementaron las **Propuestas C (ST-GRU)** y **D (TCN Multi-Output)** tal como las describe la literatura: serie real normalizada sin filtro externo, sin máscara de saltos, misma semilla y mismo protocolo de entrenamiento.
- El grafo del ST-GRU combina vecindad física (celdas en serie) y similitud de comportamiento en train.
- Salidas: `models/modelo_C.pt` y `models/modelo_D.pt`. La comparación está en el notebook 07.